Demo: Teach an LLM a New Skill with SFT
Welcome! This notebook is a short demonstration to show you how to teach a Large Language Model (LLM) a new skill using Supervised Fine-Tuning (SFT).

LLMs are great at many things, but they don't know everything. Sometimes, we need to teach them a specific, new task. In this demo, we'll teach a small LLM to add the suffish "-ish" to the ends of words.

This demo follows the exact same structure as the exercise you're about to do. Pay attention to the steps, as you'll be repeating them to teach the model how to spell.

What you'll see in this demo
Setup: Import libraries and configure the environment.
Load the model: Use a small, instruction-tuned model as our starting point.
Create a dataset: Generate a simple dataset of words and their -ish variants.
Evaluate the base model: See how the model does before any training.
Configure LoRA and train: Use Parameter-Efficient Fine-Tuning (PEFT) with LoRA to train our model efficiently.
Evaluate the fine-tuned model: Test the model again to see its new skill in action.
Setup

In [2]:
import os
import torch
from datasets import Dataset
from  transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

C:\shiva\coding\python_projects\dear_comrade_data_science_zero_to_hero\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Setup

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
torch.set_num_threads(max(1, os.cpu_count()//2))
print(f"Using device: {device}")

Using device: cpu


# Step 1. Load the tokenizer and base model
## We'll use HuggingFaceTB/SmolLM2-135M-Instruct, a small model with 135 million parameters. Its small size makes it perfect for a quick demonstration on a standard computer.

In [4]:
model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
model = model.to(device)
print(f"model loaded: {model_id}")

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 19945.29it/s]


model loaded: HuggingFaceTB/SmolLM2-135M-Instruct


# Step 2. Create the dataset
## Next, we'll create a small dataset of examples to teach the model our new task.

In [5]:
DEMO_WORDS = [
    "idea", "glow", "rust", "maze", "echo", "wisp", "veto", "lush", "gaze", "knit", "fume", "plow",
    "void", "oath", "grim", "crisp", "lunar", "fable", "quest", "verge", "brawn", "elude", "aisle",
    "ember", "crave", "ivory", "mirth", "knack", "wryly", "onset", "mosaic", "velvet", "sphinx",
    "radius", "summit", "banner", "cipher", "glisten", "mantle", "scarab", "expose", "fathom",
    "tavern", "fusion", "relish", "lantern", "enchant", "torrent", "capture", "orchard", "eclipse",
    "frescos", "triumph", "absolve", "gossipy", "prelude", "whistle", "resolve", "zealous",
    "mirage", "aperture", "sapphire",
]

In [6]:
def generate_records():
    for word in DEMO_WORDS:
        prompt = (
            f"Add -ish to the end of the word.\n"
            "hello -> hello-ish\n"
            "learn -> learn-ish\n"
            f"{word} -> "
        )
        completion = f"{word}-ish"
        yield {"prompt": prompt, "completion": completion}
ds = Dataset.from_generator(generate_records)
ds = ds.train_test_split(test_size=0.2, seed=42)
print("First training example:")
print(ds["train"][0])

First training example:
{'prompt': 'Add -ish to the end of the word.\nhello -> hello-ish\nlearn -> learn-ish\nivory -> ', 'completion': 'ivory-ish'}


# Step 3. Evaluate the base model
## Before we train, let's see if the model already knows how to do this.

In [7]:
# A helper function to test the model's translation ability
def check_translation(model, tokenizer, prompt:str, actual_translation:str):
    # Prepare the input for the model
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    # Generate a response from the model
    gen = model.generate(**inputs, max_new_tokens=15, use_cache=False)
    output = tokenizer.decode(gen[0], skip_special_tokens=True)
    # Extract just the translated part
    proposed_translation = output.split("->")[-1].strip().split("\n")[0].strip()
    is_correct = actual_translation.lower() == proposed_translation.lower()
    # Check if the model's answer is correct
    is_correct = proposed_translation == actual_translation
    print(
        f"Proposed: {proposed_translation} | Actual: {actual_translation} "
        f"| Correct: {'✅' if is_correct else '❌'}"
    )
    return is_correct

In [8]:
print("--- Evaluating Base Model (Before Training) ---")
num_correct = 0
num_examples = len(ds["test"])
for example in ds['test']:
    prompt = example["prompt"]
    completion = example["completion"]
    is_correct = check_translation(model, tokenizer, prompt, completion)
    num_correct += is_correct
print(f"\nResult: {num_correct}/{num_examples} correct.")

--- Evaluating Base Model (Before Training) ---
Proposed: wryly-ish | Actual: wryly-ish | Correct: ✅
Proposed: 1-ish | Actual: glisten-ish | Correct: ❌
Proposed: quest-ish | Actual: quest-ish | Correct: ✅
Proposed: ire-ish | Actual: crave-ish | Correct: ❌
Proposed: ils-ish | Actual: lush-ish | Correct: ❌
Proposed: файлей | Actual: fable-ish | Correct: ❌
Proposed: knack-ish | Actual: knack-ish | Correct: ✅
Proposed: iumph-ish | Actual: triumph-ish | Correct: ❌
Proposed: sapphire-ish | Actual: sapphire-ish | Correct: ✅
Proposed: expose-ish | Actual: expose-ish | Correct: ✅
Proposed: ils-es | Actual: frescos-ish | Correct: ❌
Proposed: wisp-ish | Actual: wisp-ish | Correct: ✅
Proposed: mi-rage | Actual: mirage-ish | Correct: ❌

Result: 6/13 correct.


Step 4. Configure LoRA and train the model
We'll use Low-Rank Adaptation (LoRA) to make training fast and memory-efficient. LoRA adds a small number of new, trainable parameters to the model, freezing the original ones. This means we only have to update a tiny fraction of the model's weights.

In [9]:
lora_config = LoraConfig(
    r=64, # Rank of the update matrices. Lower is fewer parameters.
    lora_alpha=16, # LoRA scaling factor. Generally set to 16.
    lora_dropout=0.05,  # Dropout for LoRA layers
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
trainable_params = sum(param.numel() for param in model.parameters() if param.requires_grad)
total_params = sum(param.numel() for param in model.parameters())
print(
    f"Trainable params: {trainable_params:,} / {total_params:,} ({100 * trainable_params / total_params:.2f}%)"
)

Trainable params: 3,686,400 / 138,201,408 (2.67%)


Notice that we're only training about 2.67% of the total parameters! Now, we set the training arguments.

In [10]:
training_args = SFTConfig(
    output_dir="data/model_demo",  # Directory to save artifacts
    per_device_train_batch_size=8,  # Small batch size for demo
    gradient_accumulation_steps=2,  # Two forward and backward passes per update step
    num_train_epochs=20,  # Number of times to go through the data
    learning_rate=2e-4,  # Controls how much the model weights are updated
    logging_steps=50,  # Log training progress every 10 steps
    save_strategy="no",  # Don't save model checkpoints
    report_to=[],  # Disable reporting to services like Weights & Biases
    fp16=False,  # Use full precision (fp32) for wider compatibility
    bf16=False
)

In [11]:
# Create SFTTrainer

sftTrainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
)
print("start training model")
sftTrainer.train()

Tokenizing train dataset:   0%|          | 0/49 [00:00<?, ? examples/s][RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+completion. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+completion. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+completion. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+completion. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that 

start training model


C:\shiva\coding\python_projects\dear_comrade_data_science_zero_to_hero\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

Step 5. Evaluate the fine-tuned model
Training is done! Now for the moment of truth. Let's see if our model learned the task.

In [ ]:
print("--- Evaluating Base Model (Before Training) ---")
num_correct = 0
num_examples = len(ds["test"])
for example in ds['test']:
    prompt = example["prompt"]
    completion = example["completion"]
    is_correct = check_translation(model, tokenizer, prompt, completion)
    num_correct += is_correct
print(f"\nResult: {num_correct}/{num_examples} correct.")